In [ ]:
import pandas as pd
from pathlib import Path
# test_prediction = pd.read_parquet("/shared/share_mala/zj2398/mimic/task/regression/motor/results/bilirubin/motor/test_predictions_50k.parquet")
# task_list = ["in_hospital_mortality","long_los", "readmission","ami","masld","stroke"]
task_list = ["creatinine","platelets","pao2","glucose"]
regression_base = Path("/shared/share_mala/zj2398/mimic/task/regression")
for task in task_list:
    # llama_test_prediction = pd.read_parquet(outcome_base/task/"cc-llama/llama_298453.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    # motor_test_prediction = pd.read_parquet(outcome_base/task/"motor/test_predictions/predictions.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    # mtpp_test_prediction = pd.read_parquet(outcome_base/task/"transformer_mtpp/test_predictions/predictions.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    # check_label()
    llama_test_prediction = pd.read_parquet(regression_base/"cc_llama/results"/task/"cc_llama"/"test_predictions.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    motor_test_prediction = pd.read_parquet(regression_base/"motor/results"/task/"motor"/"test_predictions_50k.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    mtpp_test_prediction = pd.read_parquet(regression_base/"transformer_mtpp/results"/task/"transformer_mtpp"/"test_predictions_50k.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    # check_label()
    print(mtpp_test_prediction)
    print(llama_test_prediction)
    print(motor_test_prediction)

       subject_id     prediction_time  y_true    y_pred  residual
0        10003637 2150-05-16 22:17:00     3.0  3.316756 -0.316756
1        10003637 2150-05-17 11:27:00     2.9  2.975724 -0.075724
2        10003637 2150-05-17 21:55:00     3.0  3.398387 -0.398387
3        10003637 2150-05-18 21:48:00     4.1  3.558089  0.541911
4        10003637 2150-05-19 10:00:00     4.7  3.992475  0.707525
...           ...                 ...     ...       ...       ...
49995    19998330 2178-10-12 01:08:00     2.4  1.701636  0.698364
49996    19998330 2178-10-21 23:33:00     2.1  2.506529 -0.406529
49997    19998330 2178-10-22 10:42:00     2.1  2.118041 -0.018041
49998    19998330 2178-10-22 23:34:00     2.3  2.428836 -0.128836
49999    19998330 2178-11-29 00:05:00     1.9  1.627950  0.272050

[50000 rows x 5 columns]
       subject_id     prediction_time  y_true    y_pred  residual
0        10003637 2150-05-16 22:17:00     3.0  2.179926  0.820074
1        10003637 2150-05-17 11:27:00     2.0  2.0

In [ ]:
import pandas as pd
import numpy as np
task_list = ["creatinine","platelets","pao2","glucose"]

def check_labels(mtpp, llama, motor):
    """Assert all three DataFrames have identical subject_id, prediction_time, y_true."""
    for col in ["subject_id", "prediction_time", "y_true"]:
        assert col in mtpp.columns and col in llama.columns and col in motor.columns, \
            f"Column '{col}' missing in one of the DataFrames"
    assert len(mtpp) == len(llama) == len(motor), \
        f"Length mismatch: mtpp={len(mtpp)}, llama={len(llama)}, motor={len(motor)}"
    assert (mtpp["subject_id"].values == llama["subject_id"].values).all(), \
        "subject_id mismatch between mtpp and llama"
    assert (mtpp["subject_id"].values == motor["subject_id"].values).all(), \
        "subject_id mismatch between mtpp and motor"
    assert (mtpp["prediction_time"].values == llama["prediction_time"].values).all(), \
        "prediction_time mismatch between mtpp and llama"
    assert (mtpp["prediction_time"].values == motor["prediction_time"].values).all(), \
        "prediction_time mismatch between mtpp and motor"
    for name, other in [("llama", llama), ("motor", motor)]:
    # for name, other in [("motor", motor)]:
        diff = np.abs(mtpp["y_true"].values - other["y_true"].values)
        max_diff = np.max(diff)
        print(f"  max |y_true| diff — mtpp vs {name}: {max_diff:.2e}")
        mask = diff > 1e-3
        if mask.any():
            mismatch = pd.DataFrame({
                "subject_id": mtpp["subject_id"].values[mask],
                "prediction_time": mtpp["prediction_time"].values[mask],
                "y_true_mtpp": mtpp["y_true"].values[mask],
                f"y_true_{name}": other["y_true"].values[mask],
                "abs_diff": diff[mask],
            })
            print(f"  ⚠ {len(mismatch)} rows differ (atol>1e-3) between mtpp and {name}:")
            print(mismatch.head(20).to_string(index=False))
            print("y_true mismatch between mtpp and {name} (max diff={max_diff:.2e}, n_mismatch={len(mismatch)})")
            # assert False, f"y_true mismatch between mtpp and {name} (max diff={max_diff:.2e}, n_mismatch={len(mismatch)})"
    # print(f"  ✓ Labels consistent across all 3 models (n={len(mtpp)})")

for task in task_list:
    llama = pd.read_parquet(regression_base/"cc_llama/results"/task/"cc_llama"/"test_predictions.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    motor = pd.read_parquet(regression_base/"motor/results"/task/"motor"/"test_predictions_50k.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    mtpp = pd.read_parquet(regression_base/"transformer_mtpp/results"/task/"transformer_mtpp"/"test_predictions_50k.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
    check_labels(mtpp, llama, motor)

  max |y_true| diff — mtpp vs llama: 0.00e+00
  max |y_true| diff — mtpp vs motor: 0.00e+00


In [40]:
regression_label = pd.read_parquet("/shared/share_mala/zj2398/mimic/task/regression/cohort/icu_stay_4h_100k/creatinine/creatinine.parquet")
print(regression_label)

import pandas as pd
import numpy as np
from pathlib import Path

regression_base = Path("/shared/share_mala/zj2398/mimic/task/regression")


# ── Load the full parquet (150k rows) ──
# ── Load the two model prediction files ──
mtpp = pd.read_parquet(regression_base / "transformer_mtpp/results" / task / "transformer_mtpp" / "test_predictions_50k.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True]).drop_duplicates(subset=["subject_id", "prediction_time"],keep=False)
motor = pd.read_parquet(regression_base / "motor/results" / task / "motor" / "test_predictions_50k.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True]).drop_duplicates(subset=["subject_id", "prediction_time"],keep=False)
llama = pd.read_parquet(regression_base/"cc_llama/results"/task/"cc_llama"/"test_predictions.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True]).drop_duplicates(subset=["subject_id", "prediction_time"],keep=False)

KEY_COLS = ["subject_id", "prediction_time"]

# task = "creatinine"  # change as needed

for task in ["creatinine","platelets","pao2","glucose"]:
    full_df = pd.read_parquet(f"/shared/share_mala/zj2398/mimic/task/regression/cohort/icu_stay_4h_100k/{task}/{task}.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True]).drop_duplicates(subset=["subject_id", "prediction_time"],keep=False)  # TODO: set path
    kuvira_df = pd.read_parquet(f"/shared/share_mala/zj2398/mimic/task/regression/kuvira_icu_stay_4h_100k/{task}/{task}.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True]).drop_duplicates(subset=["subject_id", "prediction_time"],keep=False)  # TODO: set path

    # print(kuvira_df)
    merged = kuvira_df.merge(full_df, on=KEY_COLS, how="left", indicator=True)

    # Check 1: every row in pred_df has a match in full_df
    missing = merged[merged["_merge"] == "left_only"]
    if len(missing) > 0:
        print(f"⚠  {len(missing)} rows NOT found in full_df:")
        print(missing[KEY_COLS].head(10).to_string(index=False))
    else:
        print(f"✓: all {len(kuvira_df)} rows found in full_df")

    # Check 2: y_true == numerical_value where matched
    matched = merged[merged["_merge"] == "both"]
    # print(matched)
    diff = np.abs(matched["numerical_value_x"].values - matched["numerical_value_y"].values)
    max_diff = np.max(diff) if len(diff) > 0 else 0
    n_mismatch = np.sum(diff > 1e-3)
    print(f"  max |numerical_value - numerical_value|: {max_diff:.2e}, rows with diff>1e-3: {n_mismatch}")
    # if n_mismatch > 0:
    #     bad = matched[diff > 1e-3][KEY_COLS + ["y_true", "numerical_value"]].head(10)
    #     print(bad.to_string(index=False))
    print()

    # for name, pred_df in [("mtpp", mtpp), ("llama", llama)]:
    #     merged = pred_df.merge(full_df, on=KEY_COLS, how="left", indicator=True)

    #     # Check 1: every row in pred_df has a match in full_df
    #     missing = merged[merged["_merge"] == "left_only"]
    #     if len(missing) > 0:
    #         print(f"⚠ {name}: {len(missing)} rows NOT found in full_df:")
    #         print(missing[KEY_COLS].head(10).to_string(index=False))
    #     else:
    #         print(f"✓ {name}: all {len(pred_df)} rows found in full_df")

    #     # Check 2: y_true == numerical_value where matched
    #     matched = merged[merged["_merge"] == "both"]
    #     diff = np.abs(matched["y_true"].values - matched["numerical_value"].values)
    #     max_diff = np.max(diff) if len(diff) > 0 else 0
    #     n_mismatch = np.sum(diff > 1e-3)
    #     print(f"  max |y_true - numerical_value|: {max_diff:.2e}, rows with diff>1e-3: {n_mismatch}")
    #     # if n_mismatch > 0:
    #     #     bad = matched[diff > 1e-3][KEY_COLS + ["y_true", "numerical_value"]].head(10)
    #     #     print(bad.to_string(index=False))
    #     print()

        subject_id     prediction_time               code  numerical_value
0         14338016 2160-01-12 19:50:00  LAB//50912//mg/dL              1.3
1         18962474 2177-09-15 03:08:00  LAB//50912//mg/dL              0.6
2         15225520 2166-11-24 18:56:00  LAB//50912//mg/dL              1.6
3         14267492 2153-03-14 22:16:00  LAB//50912//mg/dL              0.9
4         10569882 2193-05-19 15:33:00  LAB//50912//mg/dL              0.9
...            ...                 ...                ...              ...
149995    18175863 2187-09-15 23:09:00  LAB//50912//mg/dL              2.0
149996    10659096 2135-11-25 07:09:00  LAB//50912//mg/dL              1.2
149997    12213423 2134-03-03 22:40:00  LAB//50912//mg/dL              1.7
149998    16175900 2169-01-09 20:23:00  LAB//50912//mg/dL              2.2
149999    15121546 2145-03-30 02:20:00  LAB//50912//mg/dL              1.4

[150000 rows x 4 columns]
✓: all 149946 rows found in full_df
  max |numerical_value - numerical_va

In [24]:
import pandas as pd
for task in ["creatinine","platelets","pao2","glucose"]:
# for task in ["creatinine"]:
    base_dir = f"/shared/share_mala/zj2398/mimic/task/regression/cohort/icu_stay_4h_100k/{task}/{task}.parquet"
    df = pd.read_parquet(base_dir)

    # Find rows where (subject_id, prediction_time) pair is duplicated
    duplicates = df[df.duplicated(subset=["subject_id", "prediction_time"], keep=False)]

    # Sort so duplicates are grouped together for easy comparison
    duplicates = duplicates.sort_values(["subject_id", "prediction_time"])

    print(f"Total duplicate rows: {len(duplicates)}")
    print(f"Unique duplicated (subject_id, prediction_time) pairs: {duplicates.groupby(['subject_id', 'prediction_time']).ngroups}")
    print(duplicates.to_string())

Total duplicate rows: 54
Unique duplicated (subject_id, prediction_time) pairs: 27
        subject_id     prediction_time               code  numerical_value
125124    10068474 2186-02-09 01:10:00  LAB//50912//mg/dL              1.9
139285    10068474 2186-02-09 01:10:00  LAB//50912//mg/dL              1.7
114431    10627213 2176-07-04 21:50:00  LAB//50912//mg/dL              0.5
138895    10627213 2176-07-04 21:50:00  LAB//50912//mg/dL              0.3
125935    10699336 2158-03-26 22:04:00  LAB//50912//mg/dL              3.1
140222    10699336 2158-03-26 22:04:00  LAB//50912//mg/dL              3.3
121614    11062332 2132-08-15 18:09:00  LAB//50912//mg/dL              3.5
146295    11062332 2132-08-15 18:09:00  LAB//50912//mg/dL              3.0
3348      12065060 2148-05-14 22:13:00  LAB//50912//mg/dL              1.0
80235     12065060 2148-05-14 22:13:00  LAB//50912//mg/dL              0.8
46496     12386418 2128-01-27 08:54:00  LAB//50912//mg/dL              3.3
55270     1238641

In [31]:
import meds_reader
from datetime import datetime
meds_reader_path = "/shared/share_mala/zj2398/mimic/data/meds_v0.6_reader"
database = meds_reader.SubjectDatabase(meds_reader_path)
for subject_id in database:
    if subject_id == 10068474:
        for event in database[subject_id].events:
            if event.time is not None and  datetime(2186, 2, 9, 6, 10, 0) >= event.time >= datetime(2186, 2, 9, 1, 10, 0) and event.code == "LAB//50912//mg/dL":
                print(event,event.time,event.numeric_value)
        break

Event(time=2186-02-09 02:49:00, code=LAB//50912//mg/dL, ...) 2186-02-09 02:49:00 1.899999976158142
Event(time=2186-02-09 05:10:00, code=LAB//50912//mg/dL, ...) 2186-02-09 05:10:00 1.899999976158142
Event(time=2186-02-09 05:10:00, code=LAB//50912//mg/dL, ...) 2186-02-09 05:10:00 1.7000000476837158


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed

# ── Config ──
task_list = ["in_hospital_mortality", "long_los", "readmission", "ami", "masld", "stroke"]
outcome_base = Path("/shared/share_mala/zj2398/mimic/task/patient_outcome_task/results")
N_BOOTSTRAP = 5000
SEED = 42
N_JOBS = 32  # number of parallel workers


def check_labels(mtpp, llama, motor):
    """Assert all three DataFrames have identical subject_id, prediction_time, boolean_value."""
    for col in ["subject_id", "prediction_time", "boolean_value"]:
        assert col in mtpp.columns and col in llama.columns and col in motor.columns, \
            f"Column '{col}' missing in one of the DataFrames"
    assert len(mtpp) == len(llama) == len(motor), \
        f"Length mismatch: mtpp={len(mtpp)}, llama={len(llama)}, motor={len(motor)}"
    assert (mtpp["subject_id"].values == llama["subject_id"].values).all(), \
        "subject_id mismatch between mtpp and llama"
    assert (mtpp["subject_id"].values == motor["subject_id"].values).all(), \
        "subject_id mismatch between mtpp and motor"
    assert (mtpp["prediction_time"].values == llama["prediction_time"].values).all(), \
        "prediction_time mismatch between mtpp and llama"
    assert (mtpp["prediction_time"].values == motor["prediction_time"].values).all(), \
        "prediction_time mismatch between mtpp and motor"
    assert (mtpp["boolean_value"].values == llama["boolean_value"].values).all(), \
        "boolean_value mismatch between mtpp and llama"
    assert (mtpp["boolean_value"].values == motor["boolean_value"].values).all(), \
        "boolean_value mismatch between mtpp and motor"
    print(f"  ✓ Labels consistent across all 3 models (n={len(mtpp)})")


def _one_bootstrap(seed, labels, preds_a, preds_b):
    """Single bootstrap iteration — designed for joblib parallelism."""
    rng = np.random.RandomState(seed)
    n = len(labels)
    idx = rng.randint(0, n, size=n)
    y = labels[idx]
    if y.sum() == 0 or y.sum() == n:
        return np.nan
    return roc_auc_score(y, preds_a[idx]) - roc_auc_score(y, preds_b[idx])


def bootstrap_auroc_test(labels, preds_a, preds_b, n_bootstrap=N_BOOTSTRAP, seed=SEED, n_jobs=N_JOBS):
    """
    Bootstrap test: H0: AUROC_a <= AUROC_b, H1: AUROC_a > AUROC_b.
    Returns: observed_diff, ci_lower, ci_upper, p_value
    """
    observed_diff = roc_auc_score(labels, preds_a) - roc_auc_score(labels, preds_b)

    seeds = np.random.RandomState(seed).randint(0, 2**31, size=n_bootstrap)
    diffs = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_bootstrap)(s, labels, preds_a, preds_b) for s in seeds
    ))

    diffs = diffs[~np.isnan(diffs)]
    ci_lower, ci_upper = np.percentile(diffs, [2.5, 97.5])
    p_value = np.mean(diffs <= 0)
    return observed_diff, ci_lower, ci_upper, p_value


# ── Main ──
results = []

for task in task_list:
    print(f"\n{'='*60}\nTask: {task}\n{'='*60}")
    llama = pd.read_parquet(outcome_base / task / "cc-llama/llama_298453.parquet") \
        .sort_values(["subject_id", "prediction_time"]).reset_index(drop=True)
    motor = pd.read_parquet(outcome_base / task / "motor/test_predictions/predictions.parquet") \
        .sort_values(["subject_id", "prediction_time"]).reset_index(drop=True)
    mtpp = pd.read_parquet(outcome_base / task / "transformer_mtpp/test_predictions/predictions.parquet") \
        .sort_values(["subject_id", "prediction_time"]).reset_index(drop=True)

    check_labels(mtpp, llama, motor)

    labels = mtpp["boolean_value"].values.astype(int)
    mtpp_preds = mtpp["predicted_boolean_probability"].values  # adjust column name if needed
    llama_preds = llama["predicted_boolean_probability"].values
    motor_preds = motor["predicted_boolean_probability"].values

    # Point AUROCs
    auc_mtpp = roc_auc_score(labels, mtpp_preds)
    auc_llama = roc_auc_score(labels, llama_preds)
    auc_motor = roc_auc_score(labels, motor_preds)
    print(f"  AUROC — mtpp: {auc_mtpp:.4f}, llama: {auc_llama:.4f}, motor: {auc_motor:.4f}")

    # MTPP vs LLaMA
    diff, ci_lo, ci_hi, p = bootstrap_auroc_test(labels, mtpp_preds, llama_preds)
    # sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    sig = "*" if p < 0.05 else ""
    print(f"  MTPP vs LLaMA: Δ={diff:+.4f}, 95% CI=[{ci_lo:+.4f}, {ci_hi:+.4f}], p={p:.4f} {sig}")
    results.append(dict(task=task, comparison="mtpp_vs_llama", diff=diff, ci_lo=ci_lo, ci_hi=ci_hi, p=p))

    # MTPP vs MOTOR
    diff, ci_lo, ci_hi, p = bootstrap_auroc_test(labels, mtpp_preds, motor_preds)
    sig = "*" if p < 0.05 else ""
    # sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"  MTPP vs MOTOR: Δ={diff:+.4f}, 95% CI=[{ci_lo:+.4f}, {ci_hi:+.4f}], p={p:.4f} {sig}")
    results.append(dict(task=task, comparison="mtpp_vs_motor", diff=diff, ci_lo=ci_lo, ci_hi=ci_hi, p=p))

# ── Summary table ──
df_results = pd.DataFrame(results)
print(f"\n{'='*60}\nSummary\n{'='*60}")
print(df_results.to_string(index=False, float_format="%.4f"))


Task: in_hospital_mortality
  ✓ Labels consistent across all 3 models (n=33139)
  AUROC — mtpp: 0.9652, llama: 0.9003, motor: 0.9541


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

for task in ["glucose"]:
# task_list = ["creatinine","platelets","pao2","glucose"]

    feature_path = Path(f"/data/results/zj2398/femr/mimic/regression/cc_llama_old/{task}/features_with_label")
    label_path = Path(f"/data2/processed_datasets/mimic/regression/icu_stay_4h_100k_split/{task}")
    # for split in ["train","tuning","held_out"]:
    for split in ["held_out"]:
        feature_label = pd.read_parquet(feature_path/ f"{split}.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
        label = pd.read_parquet(label_path/ f"{split}.parquet").sort_values(["subject_id", "prediction_time"], ascending=[True, True])
        # print(feature_label)
        # print(label)
        print(len(label),len(feature_label))
        diff = np.abs(feature_label["numerical_value"].values - label["numerical_value"].values)
        max_diff = np.max(diff)
        print(f"  max |y_true| diff : {max_diff:.2e}")
        mask = diff > 1e-3
        if mask.any():
            mismatch = pd.DataFrame({
                "subject_id": feature_label["subject_id"].values[mask],
                "prediction_time": feature_label["prediction_time"].values[mask],
                "y_true_llama": feature_label["numerical_value"].values[mask],
                f"y_true": label["numerical_value"].values[mask],
                "abs_diff": diff[mask],
            })
            print(f"  ⚠ {len(mismatch)} rows differ (atol>1e-3)")
            # print(mismatch.head(20).to_string(index=False))
            # print("y_true mismatch between mtpp and {name} (max diff={max_diff:.2e}, n_mismatch={len(mismatch)})")

50000 50000
  max |y_true| diff : 0.00e+00


I want to do statistical testing to 